# Evaluate descriptive metadata

Faithfulness and similarity use the same means and standard errors as `03-TvdP-validate-metadata.ipynb`. Scores come from `outputs/pangaea/df_scores_pangaea_descriptive_with_baselines.csv`.

Single LLM is scored from `/home/com3dian/Github/metadata_agent/outputs/single_llm_pangaea_*`, and ReAct from `react_pangaea_*` in the same directory, excluding the smoke runs. That cell adds only those columns. Existing M³, AutoDDG, and DataAtlas scores are left as they are. A dataset with no successful file counts as zero, as for the other baselines.

The next cells print every model, then the selected-model table, then the full table.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

from langchain_openai import ChatOpenAI
from ragas import SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import FactualCorrectness, RubricsScore

METADATA_AGENT_ROOT = Path('/home/com3dian/Github/metadata_agent')
EVALUATOR_REPO_ROOT = Path('/home/com3dian/Github/process_data_metadata_agent')
ENV_PATH = EVALUATOR_REPO_ROOT / '.env'
load_dotenv(ENV_PATH, override=True)

SURF_API_KEY = os.getenv('SURF_API_KEY')
SURF_API_BASE = os.getenv('SURF_API_BASE') or 'https://willma.surf.nl/api/v0/'
if not SURF_API_KEY:
    raise RuntimeError(f'SURF_API_KEY is missing from {ENV_PATH}')

SINGLE_LLM_ROOT = METADATA_AGENT_ROOT / 'outputs'
DATASET_ROOT = EVALUATOR_REPO_ROOT / 'outputs' / 'pangaea' / 'pangaea_datasets'
SCORES_PATH = EVALUATOR_REPO_ROOT / 'outputs' / 'pangaea' / 'df_scores_pangaea_descriptive_with_baselines.csv'
CHECKPOINT_PATH = EVALUATOR_REPO_ROOT / 'outputs' / 'pangaea' / 'df_scores_single_llm_checkpoint.csv'
ANNOTATION_PATH = EVALUATOR_REPO_ROOT / 'outputs' / 'pangaea' / 'sample_20260629_163144_filtered_top5_per_topic.csv'
JUDGE_MODEL = 'mistralai/Devstral-Small-2-24B-Instruct-2512'
COL_NAMES = ['name', 'description', 'keywords']
SINGLE_LLM_RUNS = [
    'single_llm_pangaea_default-text-large',
    'single_llm_pangaea_mistralai_Devstral-Small-2-24B-Instruct-2512',
    'single_llm_pangaea_mistralai_Mistral-Small-3.2-24B-Instruct-2506',
    'single_llm_pangaea_Qwen_Qwen3.6-35B-A3B-FP8',
    'single_llm_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4',
]
REACT_CHECKPOINT_PATH = EVALUATOR_REPO_ROOT / 'outputs' / 'pangaea' / 'df_scores_react_checkpoint.csv'
REACT_RUNS = [
    'react_pangaea_default-text-large',
    'react_pangaea_mistralai_Devstral-Small-2-24B-Instruct-2512',
    'react_pangaea_mistralai_Mistral-Small-3.2-24B-Instruct-2506',
    'react_pangaea_Qwen_Qwen3.6-35B-A3B-FP8',
    'react_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4',
]

llm = LangchainLLMWrapper(
    ChatOpenAI(
        model=JUDGE_MODEL,
        api_key=SURF_API_KEY,
        base_url=SURF_API_BASE,
        default_headers={'X-API-KEY': SURF_API_KEY},
        temperature=0,
        max_tokens=2048,
        timeout=120,
        max_retries=1,
    )
)


def serialize_dataset(df):
    schema = '\n'.join(f'- {column} ({dtype})' for column, dtype in df.dtypes.items())
    sample = df.head(5).to_markdown(index=False)
    return f'Columns:\n{schema}\n\nSample rows:\n{sample}\n\nRow count: {len(df)}'


def format_keywords(value):
    if isinstance(value, list):
        return str(value)
    return value


def load_predictions(folder_path: Path, file_glob: str) -> dict:
    predictions = {}
    for path in sorted(folder_path.glob(file_glob)):
        with path.open(encoding='utf-8') as handle:
            result = json.load(handle)
        if result.get('status') != 'ok':
            continue
        metadata = result.get('metadata') or {}
        fields = {column: metadata.get(column) for column in COL_NAMES}
        fields['keywords'] = format_keywords(fields['keywords'])
        if any(not isinstance(fields[column], str) or not fields[column].strip() for column in COL_NAMES):
            raise ValueError(f'Missing descriptive field in {path}')
        dataset_id = int(result.get('dataset_id', path.name.split('_')[0]))
        if dataset_id in predictions:
            raise ValueError(f'Duplicate dataset {dataset_id} in {folder_path}')
        predictions[dataset_id] = fields
    if not predictions:
        raise ValueError(f'No predictions found in {folder_path}')
    return predictions


grounding_scorer = RubricsScore(
    rubrics={
        'score1_description': 'Makes claims that contradict the data or reference fields/values that do not exist in it.',
        'score2_description': 'Several claims are unsupported by the data or appear invented.',
        'score3_description': 'Mostly supportable, but contains at least one unsupported or speculative claim.',
        'score4_description': 'Every claim is supportable from the data; some claims are vague or generic.',
        'score5_description': 'Every claim is directly supported by the data and is specific to it.',
    },
    llm=llm,
)
similarity_scorer = FactualCorrectness(llm=llm, mode='precision', atomicity='low')


async def score_one(row_id, predicted, reference_description):
    dataset = pd.read_csv(DATASET_ROOT / f'{int(row_id)}.csv')
    data_context = serialize_dataset(dataset)
    predicted_abstract = '\n\n'.join(f'{key}: {predicted[key]}' for key in COL_NAMES)
    grounding_sample = SingleTurnSample(
        user_input=(
            'A model generated metadata for a dataset from the data alone.\n\n'
            f'DATASET:\n{data_context}\n\n'
            'Judge only whether the TITLE, DESCRIPTION, and KEYWORDS below are grounded in this data — '
            'whether its claims could be inferred from the data. Do not reward or '
            'penalize style, completeness, or omissions.'
        ),
        response=predicted_abstract,
    )
    similarity_sample = SingleTurnSample(
        response=predicted['description'],
        reference=reference_description,
    )
    faithfulness = await grounding_scorer.single_turn_ascore(grounding_sample)
    similarity = await similarity_scorer.single_turn_ascore(similarity_sample)
    return (faithfulness - 1) / 4, similarity


async def score_added_runs(run_folders, name_prefix, column_prefix, file_glob, checkpoint_path):
    original_scores = pd.read_csv(SCORES_PATH)
    annotations = pd.read_csv(ANNOTATION_PATH)
    df_scores = original_scores.copy()
    if checkpoint_path.exists():
        checkpoint = pd.read_csv(checkpoint_path).set_index('id')
        for column in checkpoint.columns:
            if column_prefix not in column:
                continue
            df_scores[column] = df_scores['id'].map(checkpoint[column])

    for run_folder in run_folders:
        model_key = run_folder.removeprefix(name_prefix)
        column_key = f'{column_prefix}{model_key}'
        faith_column = f'faith_{column_key}'
        similarity_column = f'sim_{column_key}'
        predictions = load_predictions(SINGLE_LLM_ROOT / run_folder, file_glob)
        unknown_ids = set(predictions) - set(df_scores['id'])
        if unknown_ids:
            raise ValueError(f'Prediction IDs absent from the score table: {sorted(unknown_ids)}')

        if faith_column not in df_scores.columns:
            df_scores[faith_column] = pd.NA
        if similarity_column not in df_scores.columns:
            df_scores[similarity_column] = pd.NA

        pending = []
        for _, score_row in df_scores.iterrows():
            dataset_id = int(score_row['id'])
            already_scored = pd.notna(score_row[faith_column]) and pd.notna(score_row[similarity_column])
            if already_scored:
                continue
            if dataset_id not in predictions:
                df_scores.loc[df_scores['id'] == dataset_id, faith_column] = 0.0
                df_scores.loc[df_scores['id'] == dataset_id, similarity_column] = 0.0
                continue
            pending.append(dataset_id)

        print(f'{column_key}: {len(predictions)} predictions, {len(pending)} still to score')
        annotation_by_id = annotations.set_index('id')
        for dataset_id in tqdm(pending, desc=column_key):
            try:
                faithfulness, similarity = await score_one(
                    dataset_id,
                    predictions[dataset_id],
                    annotation_by_id.loc[dataset_id, 'description'],
                )
            except Exception as error:
                print(f'Row {dataset_id} failed: {error}')
                continue
            df_scores.loc[df_scores['id'] == dataset_id, faith_column] = faithfulness
            df_scores.loc[df_scores['id'] == dataset_id, similarity_column] = similarity
            checkpoint_frame = original_scores.copy()
            for column in df_scores.columns:
                if column_prefix in column:
                    checkpoint_frame[column] = df_scores[column]
            checkpoint_frame.to_csv(checkpoint_path, index=False)

        remaining = int(df_scores[faith_column].isna().sum() + df_scores[similarity_column].isna().sum())
        if remaining:
            raise RuntimeError(f'{column_key} still has {remaining} missing scores; final table was not updated')

    final_scores = original_scores.copy()
    for column in df_scores.columns:
        if column_prefix in column:
            final_scores[column] = df_scores[column].to_numpy()
    final_scores.to_csv(checkpoint_path, index=False)
    final_scores.to_csv(SCORES_PATH, index=False)
    print(f'Saved {column_prefix} columns to {SCORES_PATH}')

await score_added_runs(
    SINGLE_LLM_RUNS,
    'single_llm_pangaea_',
    'SingleLLM_',
    '*_single_llm.json',
    CHECKPOINT_PATH,
)
await score_added_runs(
    REACT_RUNS,
    'react_pangaea_',
    'ReAct_',
    '*_react.json',
    REACT_CHECKPOINT_PATH,
)


/home/com3dian/miniconda3/envs/sam2/lib/python3.12/site-packages/instructor/providers/gemini/client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]


SingleLLM_default-text-large: 70 predictions, 0 still to score


/tmp/ipykernel_137143/3752141835.py:12: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import FactualCorrectness, RubricsScore
/tmp/ipykernel_137143/3752141835.py:12: DeprecationWarning: Importing RubricsScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import RubricsScore
  from ragas.metrics import FactualCorrectness, RubricsScore
/tmp/ipykernel_137143/3752141835.py:46: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  llm = LangchainLLMWrapper(


SingleLLM_default-text-large: 0it [00:00, ?it/s]

SingleLLM_mistralai_Devstral-Small-2-24B-Instruct-2512: 65 predictions, 0 still to score


SingleLLM_mistralai_Devstral-Small-2-24B-Instruct-2512: 0it [00:00, ?it/s]

... (5 outputs left out)

SingleLLM_Sehyo_Qwen3.5-122B-A10B-NVFP4: 0it [00:00, ?it/s]

Saved SingleLLM_ columns to /home/com3dian/Github/process_data_metadata_agent/outputs/pangaea/df_scores_pangaea_descriptive_with_baselines.csv
ReAct_default-text-large: 75 predictions, 75 still to score


ReAct_default-text-large:   0%|          | 0/75 [00:00<?, ?it/s]

Row 961119 failed: Request timed out.


In [ ]:
from pathlib import Path

import pandas as pd

SCORES_PATH = Path('/home/com3dian/Github/process_data_metadata_agent/outputs/pangaea/df_scores_pangaea_descriptive_with_baselines.csv')
DISPLAY_NAMES = {
    'llama3_70B': 'Llama3 70B',
    'default-text-large': 'Llama3 70B',
    'Qwen_3.6_35B': 'Qwen 3.6 35B',
    'Qwen_Qwen3.6-35B-A3B-FP8': 'Qwen 3.6 35B',
    'Qwen3.5-122B': 'Qwen 3.5 122B',
    'Sehyo_Qwen3.5-122B-A10B-NVFP4': 'Qwen 3.5 122B',
    'mistralai_Devstral-Small-2-24B-Instruct-2512': 'Mistral Devstral 2 24B',
    'mistralai_Mistral-Small-3.2-24B-Instruct-2506': 'Mistral Small 3.2 24B',
}

def model_keys(df_scores, prefix):
    keys = []
    for column in df_scores.columns:
        if not column.startswith('faith_'):
            continue
        key = column.removeprefix('faith_')
        if prefix:
            if key.startswith(prefix):
                keys.append(key.removeprefix(prefix))
        elif not key.startswith(('AutoDDG_', 'DataAtlas_', 'SingleLLM_', 'ReAct_')):
            keys.append(key)
    return keys

def score_label(series):
    mean = round(100 * series.mean(), 1)
    sem = round(100 * series.sem(), 1)
    return f'{mean} ± {sem}'

def descriptive_results(df_scores, prefix):
    rows = []
    for key in model_keys(df_scores, prefix):
        column_key = f'{prefix}{key}'
        rows.append({
            'Model': DISPLAY_NAMES.get(key, key),
            'Faithfulness [%]': score_label(df_scores[f'faith_{column_key}']),
            'Similarity [%]': score_label(df_scores[f'sim_{column_key}']),
        })
    return pd.DataFrame(rows).sort_values('Model').reset_index(drop=True)

df_scores = pd.read_csv(SCORES_PATH)
results = {
    'AutoDDG': descriptive_results(df_scores, 'AutoDDG_'),
    'DataAtlas': descriptive_results(df_scores, 'DataAtlas_'),
    'M³': descriptive_results(df_scores, ''),
    'Single LLM': descriptive_results(df_scores, 'SingleLLM_'),
    'ReAct': descriptive_results(df_scores, 'ReAct_'),
}
for method in ['AutoDDG', 'DataAtlas', 'M³', 'Single LLM', 'ReAct']:
    print(method)
    display(results[method])


In [ ]:
# Edit these to the Model names printed above.
BEST_MODELS = {
    'M³': 'Llama3 70B',
    'AutoDDG': 'Mistral Small 3.2 24B',
    'DataAtlas': 'Qwen 3.6 35B',
}

def compare_selected_models(results, best_models):
    rows = []
    for method, model in best_models.items():
        table = results[method]
        match = table.loc[table['Model'] == model]
        if match.empty:
            available = ', '.join(table['Model'])
            raise KeyError(f'{model!r} is not a {method} model. Available: {available}')
        chosen = match.iloc[0]
        rows.append({
            'Method': method,
            'Best model': model,
            'Faithfulness [%]': chosen['Faithfulness [%]'],
            'Similarity [%]': chosen['Similarity [%]'],
        })
    return pd.DataFrame(rows)

def print_descriptive_latex(df):
    latex_df = df.copy()
    latex_df['Method'] = latex_df['Method'].replace({'M³': r'M$^3$'})
    latex = latex_df.to_latex(
        index=False,
        escape=False,
        caption='Selected model per method for PANGAEA descriptive metadata. Faithfulness and similarity are mean ± standard error, in percent.',
        label='tab:best_method_descriptive',
    )
    print(latex)
    return latex

method_comparison = compare_selected_models(results, BEST_MODELS)
print_descriptive_latex(method_comparison)
method_comparison


In [ ]:
def all_models_table(results):
    frames = []
    for method in ['M³', 'AutoDDG', 'DataAtlas', 'Single LLM', 'ReAct']:
        frame = results[method].copy()
        frame.insert(0, 'Method', method)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def print_all_models_latex(df):
    latex_df = df.copy()
    latex_df['Method'] = latex_df['Method'].replace({'M³': 'M3PLACEHOLDER'})
    grouped = latex_df.set_index(['Method', 'Model'])
    latex = grouped.to_latex(
        index=True,
        escape=True,
        multirow=True,
        column_format='llcc',
        caption=r'Faithfulness and similarity for every model of M$^3$, AutoDDG, DataAtlas, Single LLM, and ReAct on PANGAEA. Values are mean $\pm$ standard error, in percent.',
        label='tab:all_models_descriptive',
    )
    header = (
        ' &  & Faithfulness [\\%] & Similarity [\\%] \\\\\n'
        'Method & Model &  &  \\\\\n'
    )
    latex = latex.replace(
        header,
        'Method & Model & Faithfulness [\\%] & Similarity [\\%] \\\\\n',
    )
    latex = latex.replace(r'\cline{1-4}', r'\midrule')
    latex = latex.replace('\\midrule\n\\bottomrule', '\\bottomrule')
    latex = latex.replace('M3PLACEHOLDER', r'M$^3$')
    latex = latex.replace('±', r'$\pm$')
    print(latex)
    return latex


all_models = all_models_table(results)
print_all_models_latex(all_models)
all_models.set_index(['Method', 'Model'])